# Laser Dataset Inspector

Создайте `config.local.yaml` на основе шаблона в корне проекта. Запускайте JupyterLab из корня проекта. Ячейки используют существующую разметку и порядок строк CSV.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from laser_detection.config import load_config
from laser_detection.data import load_dataset
from laser_detection.export import export_inspection
from laser_detection.geometry import add_geometry
from laser_detection.statistics import dataset_summary, setup_summary
from laser_detection.visualization import inspect_image, inspection_figures

config = load_config()
frame = add_geometry(load_dataset(config))
figures = inspection_figures(frame, config)

## Обзор датасета

In [ ]:
display(dataset_summary(frame), setup_summary(frame))

## Пространственное распределение

Нажмите на сетап в легенде, чтобы скрыть или показать обе серии его точек.

In [ ]:
figures["laser_positions"].show()

## Расстояние и вертикальное положение

In [ ]:
figures["distance_vs_yc"].show()

## Распределения и смещения

Углы даны в радианах. Вертикальные пары для нормализованного угла упорядочены сверху вниз.

In [ ]:
for name in ("distance_by_setup", "dx_vs_yc", "dy_vs_yc", "angle_by_setup"):
    figures[name].show()

## Просмотр кадра

Выберите имя и нажмите «Показать кадр». Файл читается в этот момент.

In [ ]:
picker = widgets.Combobox(
    options=tuple(frame["image"]), value=frame["image"].iloc[0],
    description="Кадр", ensure_option=True,
)
show_frame = widgets.Button(description="Показать кадр")
image_output = widgets.Output()


def on_show_frame(_):
    image_output.clear_output(wait=True)
    with image_output:
        figure, details, message = inspect_image(frame, picker.value, config)
        if message:
            print(message)
        if figure is not None:
            figure.show()
        if details is not None:
            display(details.to_frame("Значение"))


show_frame.on_click(on_show_frame)
display(picker, show_frame, image_output)
if not config.images_dir.is_dir():
    print("Каталог фотографий недоступен. Анализ координат доступен.")

## Локальный экспорт

Установите `EXPORT = True` и выполните ячейку. Перед коммитом очистите выводы ноутбука.

In [ ]:
EXPORT = False
if EXPORT:
    for path in export_inspection(frame, figures, config):
        print(path)